# Reading the FLUX data folder

This notebook reads the station, OpenET and footprint files in the FLUX data folder
(`D:/FLUX`) with etcompare, using the declarations in `flux_drive.yaml` (decision D-081):

1. load the configuration with the folder as its `${FLUX}` data root;
2. take stock of what the folder holds for each station;
3. read the daily station tables;
4. discover the footprint sets, and the other footprint models in one folder;
5. index one station's OpenET rasters;
6. run the footprint extraction (optional, about 20 minutes), then look at its output.

Nothing here is an accuracy assessment. `flux_drive.yaml` declares no comparison or metric,
so the last plot is a first look at the extracted values, not a model evaluation.

Run it from a repository checkout (`examples/` or the repository root) in an environment
with `etcompare[geo,plot]` installed.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

from etcompare.adapters.station import adapt_station_daily
from etcompare.config import FOOTPRINT_FILENAME_PREFIXES, FootprintSetSpec, config_from_dict
from etcompare.io.manifest import discover_footprints, discover_openet_rasters
from etcompare.registry import SiteRegistry

logging.getLogger("etcompare").setLevel(logging.WARNING)   # quiet the JSON event log

# The FLUX data folder. If the drive moves, change this line only.
FLUX = Path("D:/FLUX")
assert FLUX.is_dir(), f"{FLUX} not found: set FLUX to the folder's current location"

# examples/flux_drive.yaml, found from examples/ or from the repository root.
CONFIG = next(p for p in (Path("flux_drive.yaml"), Path("examples/flux_drive.yaml"))
              if p.exists()).resolve()

## 1. The configuration and its data root

`flux_drive.yaml` starts every data path with `${FLUX}`, declared once under `data_roots`.
Here the notebook's `FLUX` replaces that declaration before the file is validated, so the
configuration and the notebook always point at the same folder.

In [ ]:
raw = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
raw["data_roots"]["FLUX"] = str(FLUX)
cfg = config_from_dict(raw, base_dir=CONFIG.parent)
sites = SiteRegistry(cfg.site_aliases)

print("data roots:", {k: str(v) for k, v in cfg.data_roots.items()})
print("stations:  ", ", ".join(cfg.stations))
print("e.g.       ", cfg.station_tables[0].path)
print("           ", cfg.openet_rasters[0].root)
print("           ", cfg.footprint_sets[0].root)
print("outputs:   ", cfg.output_dir.resolve())

## 2. What the folder holds

One row per declared station: the daily table, the OpenET rasters per model and the
production footprints. Counts are files, not valid observations.

In [ ]:
openet_root = cfg.openet_rasters[0].root
fp_sets = {fs.set_id: fs for fs in cfg.footprint_sets}
rows = []
for site in cfg.stations:
    table = next(t for t in cfg.station_tables if t.site_id == site)
    row = {"site": site, "daily_table": table.path.exists()}
    for model in cfg.products:
        row[model] = len(list((openet_root / site / model).glob("*.tif")))
    for layer in ("etw", "mean"):
        fs = fp_sets[f"{site}_production_{layer}"]
        row[f"footprints_{layer}"] = len(list(fs.root.glob(f"ffp_{site}_daily_{layer}_*.tif")))
    rows.append(row)
inventory = pd.DataFrame(rows).set_index("site")
inventory

## 3. Daily station tables

`database/daily_output/<site>_daily.csv` (flux-data-qaqc exports), mapped to etcompare's
canonical station table. `gap_filled` marks corrected ET that was filled rather than
measured; the two are kept apart in every comparison.

In [ ]:
station = {}
summary = []
for t in cfg.station_tables:
    res = adapt_station_daily(t.path, t, cfg.stations[t.site_id])
    f = res.frame
    station[t.site_id] = f
    has_corrected = f["et_corrected_mm"].notna()
    summary.append({
        "site": t.site_id, "first": f["date"].min().date(), "last": f["date"].max().date(),
        "days": len(f), "raw_et_days": int(f["et_raw_mm"].notna().sum()),
        "corrected_measured": int((has_corrected & f["gap_filled"].eq(False)).sum()),
        "corrected_gap_filled": int((has_corrected & f["gap_filled"].eq(True)).sum()),
        "warnings": ", ".join(i.code for i in res.report if i.severity == "warning") or "-",
    })
pd.DataFrame(summary).set_index("site")

## 4. Footprint sets

Each declared set is one folder and one layer: `weighted/` holds the ET-weighted `etw`
footprints, `non_weighted/` the time-mean `mean` footprints, and US-UTM keeps both at its
top level.

In [ ]:
rows = []
for fs in cfg.footprint_sets:
    r = discover_footprints(fs, sites=sites)
    f = r.manifest.frame
    daily = f[f["period"] == "day"]
    rows.append({"set_id": fs.set_id, "folder": fs.root.relative_to(FLUX).as_posix(),
                 "daily": len(daily), "monthly": int((f["period"] == "month").sum()),
                 "first": daily["date"].min().date(), "last": daily["date"].max().date()})
pd.DataFrame(rows).set_index("set_id")

The top level of `footprints/US-UTW` also holds July 2021 footprints of other
fluxfootprints models. A footprint set takes one model, named by the filename's leading
token (`filename_prefix`), so models never mix.

In [ ]:
utw = FLUX / "footprints" / "US-UTW"
rows = []
for prefix in FOOTPRINT_FILENAME_PREFIXES:
    spec = FootprintSetSpec(f"US-UTW_{prefix}", utw, weighting_method="unverified",
                            value_semantics="normalized_cell_weight", layers=("etw",),
                            filename_prefix=prefix)
    f = discover_footprints(spec, sites=sites).manifest.frame
    daily = f[f["period"] == "day"]
    rows.append({"model": prefix, "daily_etw": len(daily),
                 "first": daily["date"].min().date(), "last": daily["date"].max().date()})
pd.DataFrame(rows).set_index("model")

## 5. OpenET rasters for one station

Indexing the whole `OpenET_Daily_Tiffs` folder takes about a minute; one station's
folder takes seconds. Filename dates show which files exist, not which pixels are valid.

In [ ]:
SITE = "US-UTJ"

src = cfg.openet_rasters[0]
idx = discover_openet_rasters(openet_root / SITE, models=cfg.model_registry(), sites=sites,
                              products=cfg.products, processing_tags=src.processing_tags,
                              on_conflict=src.on_conflict)
m = idx.manifest.frame
m.groupby("model")["date"].agg(files="count", first="min", last="max")

## 6. Footprint extraction

`etcompare run` extracts each station's ET-weighted footprint over its daily record
(about 20 minutes for the seven stations; units whose inputs did not change are reused).
Set `RUN_EXTRACTION = True` to run it from here, or run
`python -m etcompare run examples/flux_drive.yaml` in a terminal.

In [ ]:
RUN_EXTRACTION = False

if RUN_EXTRACTION:
    from etcompare.runner import run_stages

    manifest, units = run_stages(cfg, ["validate", "index", "extract"],
                                 command="notebook: flux_drive.ipynb")
    print(manifest.status, f"{len(units)} units")

## 7. A first look at the extracted values

Footprint-weighted model ET against the station's corrected ET on measured (not gap-filled)
days, one panel per model. Each panel uses every day that model has, so panels cover
different days and are **not** a fair ranking: a comparison needs a frozen cohort of common
days (`comparisons:` in the configuration).

In [ ]:
x = next(x for x in cfg.raster_extractions if x.site_id == SITE)
path = cfg.output_dir / f"{x.name}_extractions.parquet"
assert path.exists(), f"{path} not found: run the extraction first (section 6)"

ext = pd.read_parquet(path)
ext["date"] = pd.to_datetime(ext["date"])
ref = station[SITE]
measured = ref["et_corrected_mm"].notna() & ref["gap_filled"].eq(False)
ref = ref.loc[measured, ["date", "et_corrected_mm"]]
pairs = ext.merge(ref, on="date").dropna(subset=["et_mm"])

models = list(cfg.products)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5), sharex=True, sharey=True,
                         layout="constrained")
top = max(pairs["et_mm"].max(), pairs["et_corrected_mm"].max()) * 1.05
for ax, model in zip(axes.flat, models, strict=False):
    d = pairs[pairs["model"] == model]
    ax.plot([0, top], [0, top], color="#8a8985", linewidth=1, zorder=1)   # 1:1
    ax.scatter(d["et_corrected_mm"], d["et_mm"], s=10, color="#2a78d6", alpha=0.55,
               linewidths=0, zorder=2)
    ax.set_title(f"{model} (n = {len(d)})", fontsize=10, color="#0b0b0b")
    ax.grid(color="#e6e5e1", linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes.flat[-1].axis("off")
axes[0, -1].xaxis.set_tick_params(labelbottom=True)   # the panel above the empty slot
ax_lim = (0, top)
axes.flat[0].set_xlim(ax_lim)
axes.flat[0].set_ylim(ax_lim)
fig.supxlabel("Station corrected ET, measured days (mm/day)", color="#52514e")
fig.supylabel("Footprint-weighted OpenET (mm/day)", color="#52514e")
fig.suptitle(f"{SITE}: production ET-weighted footprints; grey line is 1:1", color="#0b0b0b")
plt.show()

The same pairs as a table, per model: days and the date range each covers.

In [ ]:
pairs.groupby("model").agg(days=("date", "count"), first=("date", "min"),
                           last=("date", "max"))